# 🥇 03 – Gold : modélisation en étoile (Spark SQL)

Construction d'un **schéma en étoile** en Delta Lake :

- Dimensions : `DimAthlete`, `DimTeam`, `DimGames`, `DimEvent`, `DimMedal` (dédoublonnage par `ROW_NUMBER()` + clé substitut)
- Fait : `fact_table` (jointures null-safe `<=>`) avec `hasParticipated`, `Medal_flag`, `Age`, `Height`, `Weight`
- Export de snapshots **Parquet** dans `gold/asa_export/` pour exposition à **Azure Synapse Analytics** puis **Power BI**.

In [ ]:
%run ./00_auth

In [ ]:
display(dbutils.fs.ls("abfss://silver@mohammedstorageacc2.dfs.core.windows.net/"))

In [ ]:
df = spark.read.format("delta").load("abfss://silver@mohammedstorageacc2.dfs.core.windows.net/athletes_transformed_data/")

In [ ]:
df.createOrReplaceTempView("transformed_data")

In [ ]:
%sql
SHOW STORAGE CREDENTIALS;

In [ ]:
%sql
CREATE OR REPLACE TABLE DimAthlete
USING DELTA
LOCATION "abfss://gold@mohammedstorageacc2.dfs.core.windows.net/dimensions/DimAthlete"
AS
WITH deduplicated_data AS (
  SELECT
    Name,
    Sex,
    Birth_Year,
    ROW_NUMBER() OVER (
      PARTITION BY Name, Sex, Birth_Year
      ORDER BY Name
    ) AS rn
  FROM transformed_data
),
final_table AS (
  SELECT
    Name,
    Sex,
    Birth_Year
  FROM deduplicated_data
  WHERE rn = 1
)
SELECT
  ROW_NUMBER() OVER (ORDER BY (SELECT NULL)) - 1 AS Athlete_ID,
  *
FROM final_table;


In [ ]:
%sql
CREATE OR REPLACE TABLE DimTeam 
USING DELTA
LOCATION "abfss://gold@mohammedstorageacc2.dfs.core.windows.net/dimensions/DimTeam"
AS
WITH deduplicated_data AS(
SELECT
  Team,
  NOC,
  ROW_NUMBER() OVER(
    PARTITION BY Team, NOC
    ORDER BY Team 
  ) AS rn
FROM transformed_data),

final_table AS (
SELECT
  Team,
  NOC
FROM deduplicated_data 
WHERE rn = 1
)

SELECT 
  ROW_NUMBER() OVER(
  ORDER BY(SELECT NULL)
  ) - 1 AS Team_ID,
  * 
FROM final_table

In [ ]:
%sql
CREATE OR REPLACE TABLE DimGames 
USING DELTA
LOCATION "abfss://gold@mohammedstorageacc2.dfs.core.windows.net/dimensions/DimGames"
AS
WITH deduplicated_data AS(
SELECT
  Year,
  Season,
  City,
  ROW_NUMBER() OVER(
    PARTITION BY Year, Season, City
    ORDER BY Year 
  ) AS rn
FROM transformed_data),

final_table AS (
SELECT
  Year,
  Season,
  City
FROM deduplicated_data 
WHERE rn = 1
)

SELECT 
  ROW_NUMBER() OVER(
  ORDER BY(SELECT NULL)
  ) - 1 AS Games_ID,
  * 
FROM final_table


In [ ]:
%sql
CREATE OR REPLACE TABLE DimEvent 
USING DELTA
LOCATION "abfss://gold@mohammedstorageacc2.dfs.core.windows.net/dimensions/DimEvent"
AS
WITH deduplicated_data AS(
SELECT
  Sport,
  Event,
  ROW_NUMBER() OVER(
    PARTITION BY Sport, Event
    ORDER BY Sport 
  ) AS rn
FROM transformed_data),

final_table AS (
SELECT
  Sport,
  Event
FROM deduplicated_data 
WHERE rn = 1
)

SELECT 
  ROW_NUMBER() OVER(
  ORDER BY(SELECT NULL)
  ) - 1 AS Event_ID,
  * 
FROM final_table


In [ ]:
%sql
CREATE OR REPLACE TABLE DimMedal 
USING DELTA
LOCATION "abfss://gold@mohammedstorageacc2.dfs.core.windows.net/dimensions/DimMedal"
AS
WITH deduplicated_data AS(
SELECT
  Medal,
  ROW_NUMBER() OVER(
  PARTITION BY Medal
  ORDER BY Medal 
  ) AS rn
FROM transformed_data),

final_table AS (
SELECT 
  Medal
FROM deduplicated_data 
WHERE rn = 1
)

SELECT 
  ROW_NUMBER() OVER(
  ORDER BY(SELECT NULL)
  ) - 1 AS Medal_ID,
  * 
FROM final_table
WHERE Medal IS NOT NULL


In [ ]:
%sql
CREATE OR REPLACE TABLE fact_table 
USING DELTA
LOCATION "abfss://gold@mohammedstorageacc2.dfs.core.windows.net/fact_table/"
AS 
SELECT 
  Participation_ID, 
  Athlete_ID, 
  Team_ID, 
  Games_ID, 
  Event_ID, 
  Medal_ID,
  hasParticipated, 
  Medal_flag,
  Age,
  Height,
  Weight
FROM transformed_data td LEFT JOIN dimathlete a ON td.Name <=> a.Name AND td.Sex <=> a.Sex AND td.Birth_Year <=> a.Birth_Year
LEFT JOIN dimteam t ON td.Team <=> t.TEAM AND td.NOC <=> t.NOC
LEFT JOIN dimgames g ON td.Year <=> g.Year AND td.Season <=> g.Season AND td.City <=> g.City
LEFT JOIN dimevent e ON td.Sport <=> e.Sport AND td.Event <=> e.Event
LEFT JOIN dimmedal m ON td.Medal <=> m.Medal

In [ ]:
%sql
DROP TABLE IF EXISTS DimAthlete_snapshot;

SET spark.sql.legacy.allowNonEmptyLocationInCTAS=true; CREATE TABLE DimAthlete_snapshot
USING PARQUET
LOCATION "abfss://gold@mohammedstorageacc2.dfs.core.windows.net/asa_export/DimAthlete/"
AS
SELECT * FROM DimAthlete;


In [ ]:
%sql
DROP TABLE IF EXISTS DimTeam_snapshot;

SET spark.sql.legacy.allowNonEmptyLocationInCTAS=true; CREATE TABLE DimTeam_snapshot
USING PARQUET
LOCATION "abfss://gold@mohammedstorageacc2.dfs.core.windows.net/asa_export/DimTeam"
AS
SELECT * FROM DimTeam;


In [ ]:
%sql
DROP TABLE IF EXISTS DimEvent_snapshot;

SET spark.sql.legacy.allowNonEmptyLocationInCTAS=true; CREATE TABLE DimEvent_snapshot
USING PARQUET
LOCATION "abfss://gold@mohammedstorageacc2.dfs.core.windows.net/asa_export/DimEvent"
AS
SELECT * FROM DimEvent;


In [ ]:
%sql
DROP TABLE IF EXISTS DimGames_snapshot;

SET spark.sql.legacy.allowNonEmptyLocationInCTAS=true; CREATE TABLE DimGames_snapshot
USING PARQUET
LOCATION "abfss://gold@mohammedstorageacc2.dfs.core.windows.net/asa_export/DimGames"
AS
SELECT * FROM DimGames;


In [ ]:
%sql
DROP TABLE IF EXISTS DimMedal_snapshot;

SET spark.sql.legacy.allowNonEmptyLocationInCTAS=true; CREATE TABLE DimMedal_snapshot
USING PARQUET
LOCATION "abfss://gold@mohammedstorageacc2.dfs.core.windows.net/asa_export/DimMedal"
AS
SELECT * FROM DimMedal;


In [ ]:
%sql
DROP TABLE IF EXISTS fact_table_snapshot;

SET spark.sql.legacy.allowNonEmptyLocationInCTAS=true; CREATE TABLE fact_table_snapshot
USING PARQUET
LOCATION "abfss://gold@mohammedstorageacc2.dfs.core.windows.net/asa_export/fact_table"
AS
SELECT * FROM fact_table;


In [ ]:
display(dbutils.fs.ls("abfss://gold@mohammedstorageacc2.dfs.core.windows.net/"))